In [1]:
# First of all we mount google drive for persistence
from google.colab import drive
import os

drive.mount('/content/drive')

# now we will create a project folder inside the google drive
SAVE_DIR = "/content/drive/MyDrive/GARUDAai"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Google Drive successfully mounted! Outputs will save to: {SAVE_DIR}")

Mounted at /content/drive
✅ Google Drive successfully mounted! Outputs will save to: /content/drive/MyDrive/GARUDAai


In [3]:
# check gpu acceleration (it should show Tesla T4)
!nvidia-smi

Mon Aug 24 16:28:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# now that the gpu is in place and drive is mounted now its time to install dependencies
#(-q suppresses verbose logs)
# - ultralytics: Official YOLOv8 and YOLO11 framework
# - opencv-python-headless: OpenCV optimized for cloud servers without GUI monitors
# - shapely: Geometry library for calculating virtual tripwire line intersections

!pip install -q ultralytics opencv-python-headless matplotlib shapely

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 6.5 MB/s eta 0:00:00


In [7]:

import os
import shutil
import glob
from pathlib import Path
from PIL import Image

# 1. Download the Full VisDrone Dataset (Train + Val)
from ultralytics.data.utils import download
print("Downloading Full VisDrone Dataset (Train + Val)...")

# Official download URLs used by Ultralytics
url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
download(
    [url + "VisDrone2019-DET-train.zip", url + "VisDrone2019-DET-val.zip"],
    dir="/content/datasets/VisDrone"
)

# 2. Setup Unified 3-Class Output Directory
BASE_DIR = Path("/content/border_dataset")
os.makedirs(BASE_DIR / "images" / "train", exist_ok=True)
os.makedirs(BASE_DIR / "images" / "val", exist_ok=True)
os.makedirs(BASE_DIR / "labels" / "train", exist_ok=True)
os.makedirs(BASE_DIR / "labels" / "val", exist_ok=True)

# 3. Class Remapping:
# Raw VisDrone: 1:pedestrian, 2:people -> Map to 0 (PERSON)
# Raw VisDrone: 3:bicycle, 4:car, 5:van, 6:truck, 7:tricycle, 8:awning-tricycle, 9:bus, 10:motor -> Map to 1 (VEHICLE)
VISDRONE_RAW_MAP = {
    1: 0, 2: 0, # Pedestrians / People
    3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1 # Vehicles
}

def convert_and_remap(split_name, output_split):
    print(f"\nProcessing {split_name} -> {output_split}...")

    # Locate dataset directory
    possible_roots = list(Path("/content/datasets/VisDrone").glob(f"*{split_name}*"))
    if not possible_roots:
        possible_roots = list(Path("/content").glob(f"*{split_name}*"))

    if not possible_roots:
        print(f"Could not find directory for {split_name}")
        return

    root_dir = possible_roots[0]
    img_dir = root_dir / "images"
    # Annotations could be in 'annotations' or 'labels'
    anno_dir = root_dir / "annotations" if (root_dir / "annotations").exists() else root_dir / "labels"

    print(f"   Images located at: {img_dir}")
    print(f"   Annotations at:   {anno_dir}")

    anno_files = list(anno_dir.glob("*.txt"))
    processed_count = 0

    for anno_file in anno_files:
        img_path = img_dir / (anno_file.stem + ".jpg")
        if not img_path.exists():
            continue

        # Get image dimensions to normalize coordinates
        with Image.open(img_path) as img:
            img_w, img_h = img.size

        new_labels = []
        with open(anno_file, "r") as f:
            for line in f.readlines():
                # VisDrone raw annotations are comma-separated:
                # <bbox_left>,<bbox_top>,<bbox_width>,<bbox_height>,<score>,<object_category>,<truncation>,<occlusion>
                line_clean = line.strip().replace(",", " ")
                parts = line_clean.split()
                if len(parts) < 6:
                    continue

                try:
                    bx, by, bw, bh = map(float, parts[:4])
                    cls_id = int(parts[5]) # Class is 6th element in raw VisDrone
                    score = int(parts[4])
                except ValueError:
                    continue

                # Filter out ignored regions (cls_id == 0) and low-score boxes
                if cls_id in VISDRONE_RAW_MAP and score > 0:
                    new_cls = VISDRONE_RAW_MAP[cls_id]

                    # Convert pixel (top-left x, y, w, h) to normalized (x_center, y_center, w, h)
                    xc = (bx + bw / 2.0) / img_w
                    yc = (by + bh / 2.0) / img_h
                    norm_w = bw / img_w
                    norm_h = bh / img_h

                    # Clip values between 0 and 1
                    xc = max(0.0, min(1.0, xc))
                    yc = max(0.0, min(1.0, yc))
                    norm_w = max(0.0, min(1.0, norm_w))
                    norm_h = max(0.0, min(1.0, norm_h))

                    new_labels.append(f"{new_cls} {xc:.6f} {yc:.6f} {norm_w:.6f} {norm_h:.6f}\n")

        # Save image and remapped labels
        if new_labels:
            shutil.copy(img_path, BASE_DIR / "images" / output_split / img_path.name)
            with open(BASE_DIR / "labels" / output_split / (anno_file.stem + ".txt"), "w") as f_out:
                f_out.writelines(new_labels)
            processed_count += 1

    print(f"Successfully converted and saved {processed_count} images for {output_split}!")

# Convert both Train and Validation sets
convert_and_remap("train", "train")
convert_and_remap("val", "val")

Unzipping /content/datasets/VisDrone/VisDrone2019-DET-train.zip to /content/datasets/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 625.7files/s 20.7s
Unzipping /content/datasets/VisDrone/VisDrone2019-DET-val.zip to /content/datasets/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.3Kfiles/s 0.8s

Processing train -> train...
   Images located at: /content/datasets/VisDrone/VisDrone2019-DET-train/images
   Annotations at:   /content/datasets/VisDrone/VisDrone2019-DET-train/annotations
Successfully converted and saved 6471 images for train!

Processing val -> val...
   Images located at: /content/datasets/VisDrone/VisDrone2019-DET-val/images
   Annotations at:   /content/datasets/VisDrone/VisDrone2019-DET-val/annotations
Successfully converted and saved 548 images for val!


In [8]:
import os
import shutil
from pathlib import Path
from ultralytics.data.utils import download

print("Downloading Curated Animal Dataset...")

# 1. Download animal-rich surveillance dataset (COCO128 subset)
download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip", dir="/content/datasets")

# COCO Animal Class IDs to extract:
# 14:bird, 15:cat, 16:dog, 17:horse, 18:sheep, 19:cow, 20:elephant, 21:bear
ANIMAL_CLASS_IDS = {14, 15, 16, 17, 18, 19, 20, 21}

BASE_DIR = Path("/content/border_dataset")
src_coco = Path("/content/datasets/coco128")

train_animal_count = 0
val_animal_count = 0

lbl_dir = src_coco / "labels" / "train2017"
img_dir = src_coco / "images" / "train2017"

# Split animal data: 85% into train, 15% into val
all_lbls = list(lbl_dir.glob("*.txt"))

for idx, lbl_file in enumerate(all_lbls):
    dest_split = "train" if (idx % 6 != 0) else "val" # 85/15 train-val split
    new_animal_lines = []

    with open(lbl_file, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                # If it's an animal, remap its class to 2 (animal)
                if cls_id in ANIMAL_CLASS_IDS:
                    new_animal_lines.append(f"2 {' '.join(parts[1:5])}\n")

    # If this image contained animals, copy it into our unified border dataset
    if new_animal_lines:
        img_file = img_dir / (lbl_file.stem + ".jpg")
        if img_file.exists():
            dest_img = BASE_DIR / "images" / dest_split / f"animal_{img_file.name}"
            dest_lbl = BASE_DIR / "labels" / dest_split / f"animal_{lbl_file.stem}.txt"

            shutil.copy(img_file, dest_img)
            with open(dest_lbl, "w") as f_out:
                f_out.writelines(new_animal_lines)

            if dest_split == "train":
                train_animal_count += 1
            else:
                val_animal_count += 1

print(f"\n Successfully merged animal data into border_dataset!")
print(f"   • Added to Train: {train_animal_count} animal images")
print(f"   • Added to Val:   {val_animal_count} animal images")

Unzipping /content/datasets/coco128.zip to /content/datasets/coco128...: 100% ━━━━━━━━━━━━ 263/263 3.0Kfiles/s 0.1s

 Successfully merged animal data into border_dataset!
   • Added to Train: 21 animal images
   • Added to Val:   0 animal images


In [10]:
# This script downloads a dedicated Animals in Surveillance / Wildlife dataset (~30 MB) and merges ~400+ animal images
# directly into our border_dataset
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path
print("Downloading Verified Animal Dataset in YOLO Format...")
# Download curated Animal Detection Dataset (YOLOv8/11 format)
zip_url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip"
img_url = "http://images.cocodataset.org/zips/val2017.zip"
os.makedirs("/content/datasets/coco_animals", exist_ok=True)
label_zip = "/content/datasets/coco_animals/coco_labels.zip"
img_zip = "/content/datasets/coco_animals/coco_val2017.zip"
# 1. Download Labels (~46 MB)
if not os.path.exists(label_zip):
    print("Downloading labels...")
    urllib.request.urlretrieve(zip_url, label_zip)
    with zipfile.ZipFile(label_zip, 'r') as zip_ref:
        zip_ref.extractall("/content/datasets/coco_animals")
# 2. Download Images (~780 MB - 5,000 diverse images)
if not os.path.exists(img_zip):
    print(" Downloading image package (~780 MB, takes ~15-20 secs)...")
    urllib.request.urlretrieve(img_url, img_zip)
    with zipfile.ZipFile(img_zip, 'r') as zip_ref:
        zip_ref.extractall("/content/datasets/coco_animals")
# 3. Filter Animal Classes only:
# 14:bird, 15:cat, 16:dog, 17:horse, 18:sheep, 19:cow, 20:elephant, 21:bear
COCO_ANIMAL_IDS = {14, 15, 16, 17, 18, 19, 20, 21}
BASE_DIR = Path("/content/border_dataset")
src_labels = Path("/content/datasets/coco_animals/coco/labels/val2017")
src_images = Path("/content/datasets/coco_animals/val2017")
added_train = 0
added_val = 0
print("\nExtracting animal images and converting to Class 2 (animal)...")
for idx, lbl_file in enumerate(src_labels.glob("*.txt")):
    new_animal_lines = []
    with open(lbl_file, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                if cls_id in COCO_ANIMAL_IDS:
                    # Remap to Class 2 (animal)
                    new_animal_lines.append(f"2 {' '.join(parts[1:5])}\n")
    # If this image contained an animal, add it!
    if new_animal_lines:
        img_file = src_images / (lbl_file.stem + ".jpg")
        if img_file.exists():
            dest_split = "train" if (added_train < 600) else "val" # 600 to train, rest to val

            dest_img = BASE_DIR / "images" / dest_split / f"animal_{img_file.name}"
            dest_lbl = BASE_DIR / "labels" / dest_split / f"animal_{lbl_file.stem}.txt"
            shutil.copy(img_file, dest_img)
            with open(dest_lbl, "w") as f_out:
                f_out.writelines(new_animal_lines)
            if dest_split == "train":
                added_train += 1
            else:
                added_val += 1
            # Stop after 700 images (plenty for balanced training)
            if (added_train + added_val) >= 700:
                break
print(f"\nSuccessfully injected real animal data into border_dataset!")
print(f"   • Added to Train: {added_train} animal images")
print(f"   • Added to Val:   {added_val} animal images")


Extracting animal images and converting to Class 2 (animal)...

Successfully injected real animal data into border_dataset!
   • Added to Train: 600 animal images
   • Added to Val:   100 animal images


In [12]:
# Checking final verification block
import glob

train_labels = glob.glob("/content/border_dataset/labels/train/*.txt")
class_counts = {0: 0, 1: 0, 2: 0}

for lbl in train_labels:
    with open(lbl, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) > 0:
                cls = int(parts[0])
                if cls in class_counts:
                    class_counts[cls] += 1

print("="*45)
print("FINAL UNIFIED 3-CLASS DATASET (TRAIN):")
print("="*45)
print(f"  Class 0 (PERSON)  : {class_counts[0]:,} bounding boxes")
print(f"  Class 1 (VEHICLE) : {class_counts[1]:,} bounding boxes")
print(f"  Class 2 (ANIMAL)  : {class_counts[2]:,} bounding boxes")
print("="*45)

FINAL UNIFIED 3-CLASS DATASET (TRAIN):
  Class 0 (PERSON)  : 106,396 bounding boxes
  Class 1 (VEHICLE) : 236,809 bounding boxes
  Class 2 (ANIMAL)  : 1,546 bounding boxes


In [13]:
# Creating
yaml_content = """
# Root dataset directory in Google Colab
path: /content/border_dataset
# Image folders for training and validation
train: images/train
val: images/val
# Unified 3-Class Border Taxonomy
names:
  0: person
  1: vehicle
  2: animal
"""
# Write to disk
with open("/content/border_3class.yaml", "w") as f:
    f.write(yaml_content.strip())
print("Successfully generated /content/border_3class.yaml!")
print("="*40)
print(yaml_content.strip())
print("="*40)

Successfully generated /content/border_3class.yaml!
# Root dataset directory in Google Colab
path: /content/border_dataset
# Image folders for training and validation
train: images/train
val: images/val
# Unified 3-Class Border Taxonomy
names:
  0: person
  1: vehicle
  2: animal


The Cosine Learning Rate Scheduler. It gradually slows down the learning rate as training progresses.
The Math: Final Learning Rate =
lr0
×
lrf
=
0.001
×
0.01
=
0.00001
lr0×lrf=0.001×0.01=0.00001.
Why it matters:
At the beginning, it takes bigger steps to learn fast.
At the end, it takes microscopic steps (
0.00001
0.00001) to gently settle into the absolute lowest point of the loss valley without overshooting.

mosaic = 1.0 (Mosaic Data Augmentation)  (Crucial for Surveillance)
What it means: 100% of the training images are created by stitching 4 random images into a
2
×
2
2×2 grid.
Why it’s a superpower for Border CCTV:
Shrinks objects: An intruder who was 40 pixels tall is now scaled down to 20 pixels, training the AI to spot tiny humans hundreds of meters away.
Multi-context learning: The model learns to detect objects against 4 different backgrounds at once.

mixup = 0.15 (Image Blending Augmentation)
What it means: 15% of the time, YOLO takes two different training images and blends them together with transparency (like a double-exposure photograph).
Why it matters: Real border CCTV footage has fog, desert dust, shadows, and motion blur. MixUp forces the AI to detect human and vehicle shapes even when they are semi-transparent or partially occluded.

patience = 10 (Early Stopping Trigger)
What it means: If the validation accuracy (
mAP
mAP) stops improving for 10 epochs in a row, YOLO automatically stops training early.
Why it matters:
Prevents Overfitting (memorizing training data instead of learning general patterns).
Saves your Google Colab GPU compute units if the model already peaked at Epoch 22.

In [14]:
from ultralytics import YOLO
# 1. Load the pre-trained base model
model = YOLO("yolo11s.pt")
# 2. Start Training on our 3-Class Border Dataset
results = model.train(
    data="/content/border_3class.yaml",
    epochs=35,                           # 35 epochs
    imgsz=640,                           # 640x640 resolution
    batch=16,                            # Optimal batch size for T4 GPU (16GB VRAM)
    device=0,                            # Use GPU
    optimizer="AdamW",                   # Best optimizer for fine-tuning
    lr0=0.001,                           # Initial learning rate
    lrf=0.01,                            # Cosine learning rate scheduler
    mosaic=1.0,                          # 4-image grid augmentation (Crucial for small objects)
    mixup=0.15,                          # Image blending augmentation
    patience=10,                         # Early stopping if no improvement for 10 epochs
    project="/content/drive/MyDrive/GARUDAai",  # Saves directly to your Google Drive!
    name="garuda_watchtower_yolo11s",
    save=True,
    verbose=True
)

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/border_3class.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=garuda_watchtower_yolo11s, nbs=